<a href="https://colab.research.google.com/github/udlbook/udlbook/blob/main/Notebooks/Chap13/13_2_Graph_Classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Notebook 13.2: Graph classification**

This notebook investigates representing graphs with matrices as illustrated in figure 13.4 from the book.

Work through the cells below, running each cell in turn. In various places you will see the words "TODO". Follow the instructions at these places and make predictions about what is going to happen or write code to complete the functions.

Contact me at udlbookmail@gmail.com if you find any mistakes or have any suggestions.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx

Let's build a model that maps a chemical structure to a binary decision.  This model might be used to predict whether a chemical is liquid at room temperature or not.  We'll start by drawing the chemical structure.

In [ ]:
# Define a graph that represents the chemical structure of ethanol and draw it
# Each node is labelled with the node number and the element (carbon, hydrogen, oxygen)
G = nx.Graph()
G.add_edge('0:H','2:C')
G.add_edge('1:H','2:C')
G.add_edge('3:H','2:C')
G.add_edge('2:C','5:C')
G.add_edge('4:H','5:C')
G.add_edge('6:H','5:C')
G.add_edge('7:O','5:C')
G.add_edge('8:H','7:O')
nx.draw(G, nx.spring_layout(G, seed = 0), with_labels=True, node_size=600)
plt.show()

In [ ]:
# Define adjacency matrix
# TODO -- Define the adjacency matrix for this chemical
A = np.zeros((9,9))
connections = [[0,2], [1,2], [3,2], [2,5], [4,5], [6,5], [7,5], [8,7]]
for edge in connections:
    A[edge[0], edge[1]] = 1
    A[edge[1], edge[0]] = 1

print("Adjacency Matrix A:")
print(A)

# TODO -- Define node matrix
X = np.zeros((118,9))
# Chemical numbers: Hydrogen-->1 (idx 0), Carbon-->6 (idx 5), Oxygen-->8 (idx 7)
X[0, [0,1,3,4,6,8]] = 1 # Hydrogens
X[5, [2,5]] = 1         # Carbons
X[7, [7]] = 1           # Oxygen

# Print the top 15 rows of the data matrix
print("\nNode Feature Matrix X (Top 15 elements):")
print(X[0:15,:])

Now let's define a network with four layers that maps this graph to a binary value, using the formulation in equation 13.11.

In [ ]:
# We'll need these helper functions
def ReLU(preactivation):
  return preactivation.clip(0.0)

def sigmoid(x):
  return 1.0/(1.0+np.exp(-x))

In [ ]:
K = 3; D = 200
np.random.seed(1)
Omega0 = np.random.normal(size=(D, 118)) * 2.0 / D
beta0 = np.random.normal(size=(D,1)) * 2.0 / D
Omega1 = np.random.normal(size=(D, D)) * 2.0 / D
beta1 = np.random.normal(size=(D,1)) * 2.0 / D
Omega2 = np.random.normal(size=(D, D)) * 2.0 / D
beta2 = np.random.normal(size=(D,1)) * 2.0 / D
omega3 = np.random.normal(size=(1, D))
beta3 = np.random.normal(size=(1,1))

In [ ]:
def graph_neural_network(A, X, Omega0, beta0, Omega1, beta1, Omega2, beta2, omega3, beta3):
  # Add self-connections to A
  A_hat = A + np.eye(A.shape[0])
  
  # Layer 1
  H1 = ReLU(Omega0 @ X @ A_hat + beta0)
  # Layer 2
  H2 = ReLU(Omega1 @ H1 @ A_hat + beta1)
  # Layer 3
  H3 = ReLU(Omega2 @ H2 @ A_hat + beta2)
  
  # Global Pooling (Sum across nodes)
  pooled = np.sum(H3, axis=1, keepdims=True)
  
  # Output Layer
  f = sigmoid(omega3 @ pooled + beta3)
  return f

In [ ]:
# Let's test this network
f = graph_neural_network(A,X, Omega0, beta0, Omega1, beta1, Omega2, beta2, omega3, beta3)
print("Your value is %3f: "%(f[0,0]), "True value of f: 0.310843")

In [ ]:
# Let's check permutation invariance
P = np.array([[0,1,0,0,0,0,0,0,0],
              [0,0,0,0,1,0,0,0,0],
              [0,0,0,0,0,1,0,0,0],
              [0,0,0,0,0,0,0,0,1],
              [1,0,0,0,0,0,0,0,0],
              [0,0,1,0,0,0,0,0,0],
              [0,0,0,1,0,0,0,0,0],
              [0,0,0,0,0,0,0,1,0],
              [0,0,0,0,0,0,1,0,0]]);

# TODO -- Use this matrix to permute the adjacency matrix A and node matrix X
A_permuted = P @ A @ P.T
X_permuted = X @ P.T

f = graph_neural_network(A_permuted, X_permuted, Omega0, beta0, Omega1, beta1, Omega2, beta2, omega3, beta3)
print("Permuted value is %3f: "%(f[0,0]), "True value of f: 0.310843")

### **Propanol Implementation**
Propanol has 3 carbons, 8 hydrogens, and 1 oxygen (Total 12 nodes).

In [ ]:
# Propanol: CH3-CH2-CH2-OH
# Nodes: 0,1,2 (C), 3,4,5,6,7,8,9,10 (H), 11 (O)
A_prop = np.zeros((12,12))
edges_prop = [[0,1], [1,2], [2,11], # C-C-C-O chain
              [0,3], [0,4], [0,5], # C1 Hydrogens
              [1,6], [1,7],        # C2 Hydrogens
              [2,8], [2,9],        # C3 Hydrogens
              [11,10]]             # O Hydrogen
for e in edges_prop:
    A_prop[e[0], e[1]] = 1
    A_prop[e[1], e[0]] = 1

X_prop = np.zeros((118, 12))
X_prop[0, 3:11] = 1 # H
X_prop[5, 0:3] = 1  # C
X_prop[7, 11] = 1   # O

f_prop = graph_neural_network(A_prop, X_prop, Omega0, beta0, Omega1, beta1, Omega2, beta2, omega3, beta3)
print("Propanol output value: %3f"%f_prop[0,0])
print("The GNN handled a larger graph (12 nodes vs 9) without changing parameters.")